# Tesla Vehicle Delivery Forecasting

## Project Overview

In this project, historical Tesla sales and production data from 2015 to 2025 was analyzed to predict future vehicle deliveries. The complete machine learning workflow was implemented, starting from data cleaning and exploratory analysis to model building and forecasting.

Two different approaches were used:

### 1. Machine Learning Approach

* Performed data preprocessing and feature engineering.
* Trained regression models on the dataset using a standard train-test split.
* Compared model performance using evaluation metrics.
* Applied hyperparameter tuning to improve prediction accuracy.

### 2. Time Series Forecasting Approach

* Maintained the chronological order of data.
* Created lag-based features to capture historical trends.
* Used TimeSeriesSplit for model validation.
* Forecasted future vehicle deliveries based on past patterns.

## Project Workflow

| Step                      | Description                                          |
| ------------------------- | ---------------------------------------------------- |
| Data Collection           | Loaded and inspected the Tesla dataset               |
| Data Cleaning             | Checked missing values, duplicates, and data quality |
| Exploratory Data Analysis | Analyzed trends, distributions, and relationships    |
| Feature Engineering       | Created additional useful features for modeling      |
| Model Training            | Built and evaluated regression models                |
| Hyperparameter Tuning     | Optimized model performance                          |
| Time Series Forecasting   | Predicted future deliveries using historical trends  |
| Results & Conclusion      | Compared approaches and summarized findings          |

## Key Skills Demonstrated

* Data Preprocessing
* Exploratory Data Analysis (EDA)
* Feature Engineering
* Regression Modeling
* Hyperparameter Tuning
* Time Series Forecasting
* Model Evaluation
* Data Visualization

## Conclusion

The project demonstrates an end-to-end machine learning pipeline for forecasting Tesla vehicle deliveries. Different modeling techniques were explored, and both traditional machine learning and time-series-based approaches were evaluated to understand their effectiveness in predicting future delivery numbers.


## Cell 1 — Library Imports

We load all required libraries upfront. Core dependencies include:
- **NumPy / Pandas** for numerical computation and tabular data handling
- **Matplotlib / Seaborn** for visualization
- **Scikit-learn** for preprocessing, modelling, and evaluation

In [ ]:
# Cell 1 — Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')
print('All libraries loaded successfully.')

## Cell 2 — Data Loading & Initial Inspection

We read the dataset from disk and perform a preliminary review covering:
- Shape (rows × columns)
- Descriptive statistics (mean, std, min/max)
- Column names and data types

This step confirms the dataset loaded correctly and gives us a first look at its structure before any transformations.

In [ ]:
# Cell 2 — Data Loading & Initial Inspection
raw_data = pd.read_csv('/content/tesla_deliveries_dataset_2015_2025 (1).csv')

print('── First 5 Rows ──')
display(raw_data.head())

print(f'\nDataset shape: {raw_data.shape[0]} rows × {raw_data.shape[1]} columns')

print('\n── Statistical Summary ──')
display(raw_data.describe())

print('\n── Column Names ──')
print(raw_data.columns.tolist())

print('\n── Data Types & Non-null Counts ──')
raw_data.info()

## Cell 3 — Date Column Construction

The dataset stores temporal information as separate `Year` and `Month` integer columns. We concatenate them into a single `Timestamp` column of dtype `datetime64`.

This unified datetime column will serve as the time axis for trend visualizations and, later, for sorting data chronologically in the time-series pipeline.

In [ ]:
# Cell 3 — Date Column Construction
raw_data['Timestamp'] = pd.to_datetime(
    raw_data['Year'].astype(str) + '-' + raw_data['Month'].astype(str)
)

print('Timestamp column created.')
print(f'Date range: {raw_data["Timestamp"].min().date()} → {raw_data["Timestamp"].max().date()}')

## Cell 4 — Categorical Feature Overview

We examine the three categorical columns — `Model`, `Region`, and `Source_Type` — by printing their value counts. This tells us:
- How many unique vehicle models are present
- Which geographic regions are covered
- How the data was sourced (official reports, estimates, etc.)

In [ ]:
# Cell 4 — Categorical Feature Overview
for col in ['Model', 'Region', 'Source_Type']:
    print(f'\n── {col} Value Counts ──')
    print(raw_data[col].value_counts())

## Cell 5 — Data Quality Audit

Before modelling, we verify data integrity by checking:
1. **Missing values** — any NaN per column that may require imputation
2. **Duplicate rows** — exact row copies that would inflate training samples

A clean bill of health here means we can proceed to EDA without defensive preprocessing steps.

In [ ]:
# Cell 5 — Data Quality Audit
print('── Missing Values per Column ──')
missing = raw_data.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found.')

dupe_count = raw_data.duplicated().sum()
print(f'\nDuplicate rows: {dupe_count}')

## Cells 6–18 — Exploratory Data Analysis (EDA)

EDA is the diagnostic phase of the pipeline. Before building models, we need to understand:
- How features correlate with each other and with the target
- Whether the target variable is normally distributed or skewed
- Which features show the strongest predictive relationships
- How delivery volumes vary across categories and time

The following cells produce 13 distinct visualizations covering correlations, distributions, scatter plots, categorical breakdowns, and temporal trends.

In [ ]:
# Cell 6 — Correlation Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    raw_data.corr(numeric_only=True),
    annot=True, fmt='.2f',
    cmap='RdYlGn', linewidths=0.5, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Target Variable Distribution
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(raw_data['Estimated_Deliveries'], bins=70, kde=True, color='steelblue', ax=ax)
ax.set_title('Delivery Volume Distribution', fontsize=13)
ax.set_xlabel('Estimated Deliveries')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Scatter Plots: Numerical Features vs Target
scatter_pairs = [
    ('Production_Units',   'Production Volume',      'darkorange'),
    ('Avg_Price_USD',      'Average Vehicle Price (USD)', 'mediumseagreen'),
    ('Battery_Capacity_kWh', 'Battery Capacity (kWh)', 'mediumpurple'),
    ('Range_km',           'Vehicle Range (km)',      'tomato'),
    ('Charging_Stations',  'Charging Stations',       'goldenrod'),
    ('CO2_Saved_tons',     'CO\u2082 Saved (tons)',   'limegreen'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (col, label, color) in enumerate(scatter_pairs):
    axes[i].scatter(
        raw_data[col], raw_data['Estimated_Deliveries'],
        alpha=0.4, color=color, edgecolors='none', s=18
    )
    axes[i].set_title(f'{label} vs Deliveries', fontsize=11)
    axes[i].set_xlabel(label)
    axes[i].set_ylabel('Estimated Deliveries')
    axes[i].grid(True, linestyle=':', alpha=0.4)

plt.suptitle('Numerical Features vs Estimated Deliveries', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Source Type Distribution
src_counts = raw_data['Source_Type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(src_counts.index, src_counts.values, color='cadetblue', edgecolor='white', width=0.6)
ax.set_title('Distribution of Data Source Types', fontsize=13)
ax.set_xlabel('Source Type')
ax.set_ylabel('Record Count')
for i, v in enumerate(src_counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 — Region-wise Average Deliveries (Horizontal Bar)
region_avg = raw_data.groupby('Region')['Estimated_Deliveries'].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(region_avg.index, region_avg.values, color='cornflowerblue', edgecolor='white')
ax.bar_label(bars, fmt='%.0f', padding=4, fontsize=9)
ax.set_title('Average Estimated Deliveries by Region', fontsize=13)
ax.set_xlabel('Mean Estimated Deliveries')
ax.grid(True, axis='x', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11 — Model-wise Average Deliveries
model_avg = raw_data.groupby('Model')['Estimated_Deliveries'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(model_avg.index, model_avg.values, color='salmon', edgecolor='white')
ax.set_title('Average Estimated Deliveries by Tesla Model', fontsize=13)
ax.set_xlabel('Tesla Model')
ax.set_ylabel('Mean Estimated Deliveries')
ax.grid(True, axis='y', linestyle=':', alpha=0.4)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12 — Annual Delivery Trend
yearly_avg = raw_data.groupby('Year')['Estimated_Deliveries'].mean()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(yearly_avg.index, yearly_avg.values, marker='s', color='teal', linewidth=2.5)
ax.fill_between(yearly_avg.index, yearly_avg.values, alpha=0.15, color='teal')
ax.set_title('Annual Average Delivery Trend (2015–2025)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Mean Estimated Deliveries')
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13 — Monthly Seasonality Pattern
monthly_avg = raw_data.groupby('Month')['Estimated_Deliveries'].mean()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(monthly_avg.index, monthly_avg.values, marker='o', color='coral', linewidth=2.5)
ax.set_title('Monthly Delivery Seasonality', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Mean Estimated Deliveries')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14 — Full Timeline Delivery Trend
fig, ax = plt.subplots(figsize=(13, 6))
sns.lineplot(x='Timestamp', y='Estimated_Deliveries', data=raw_data, ax=ax, color='steelblue', linewidth=1.5)
ax.set_title('Tesla Delivery Volume Over Time (2015–2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Estimated Deliveries')
ax.grid(True, linestyle=':', alpha=0.4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 15 — Boxplot: Outlier Detection
fig, ax = plt.subplots(figsize=(6, 5))
ax.boxplot(
    raw_data['Estimated_Deliveries'],
    patch_artist=True,
    boxprops=dict(facecolor='lightyellow', color='steelblue'),
    medianprops=dict(color='tomato', linewidth=2),
    whiskerprops=dict(color='steelblue'),
    capprops=dict(color='steelblue')
)
ax.set_title('Boxplot — Outlier Detection in Delivery Volumes', fontsize=12)
ax.set_ylabel('Estimated Deliveries')
ax.set_xticks([])
plt.tight_layout()
plt.show()

## Cell 16 — Pairplot: Multi-feature Relationships

A pairplot gives a quick view of bivariate relationships across key numeric features simultaneously, helping spot clusters, linear patterns, or unusual spreads.

In [ ]:
# Cell 16 — Pairplot of Key Features
pair_cols = ['Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km', 'Charging_Stations', 'Estimated_Deliveries']
sns.pairplot(raw_data[pair_cols], plot_kws=dict(alpha=0.3, s=10), diag_kind='kde')
plt.suptitle('Pairplot — Key Feature Relationships', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## Cell 17 — Violin Plot: Delivery Distribution by Region

Unlike a simple bar chart of averages, a violin plot reveals the full distribution shape — including spread, skewness, and potential multi-modality — for each region.

In [ ]:
# Cell 17 — Violin Plot: Delivery Distribution by Region
fig, ax = plt.subplots(figsize=(12, 6))
region_order = raw_data.groupby('Region')['Estimated_Deliveries'].median().sort_values(ascending=False).index
sns.violinplot(
    data=raw_data, x='Region', y='Estimated_Deliveries',
    order=region_order, palette='Set2', inner='quartile', ax=ax
)
ax.set_title('Delivery Distribution by Region', fontsize=13)
ax.set_xlabel('Region')
ax.set_ylabel('Estimated Deliveries')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Cell 18 — Box Plot: Deliveries by Tesla Model

This chart compares inter-quartile ranges across models — helping identify which vehicles drive the most consistent delivery volumes vs. high-variance outliers.

In [ ]:
# Cell 18 — Box Plot: Deliveries by Model
fig, ax = plt.subplots(figsize=(13, 6))
model_order = raw_data.groupby('Model')['Estimated_Deliveries'].median().sort_values(ascending=False).index
sns.boxplot(
    data=raw_data, x='Model', y='Estimated_Deliveries',
    order=model_order, palette='coolwarm', ax=ax
)
ax.set_title('Delivery Volume Distribution by Tesla Model', fontsize=13)
ax.set_xlabel('Tesla Model')
ax.set_ylabel('Estimated Deliveries')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Cell 19 — Outlier Removal (IQR Method)

Extreme values in the target column (`Estimated_Deliveries`) can distort both model training and evaluation metrics. We apply the **1.5 × IQR rule**:

$$\text{Lower bound} = Q_1 - 1.5 \times IQR \qquad \text{Upper bound} = Q_3 + 1.5 \times IQR$$

Records falling outside these fences are removed, producing a cleaner training set without sacrificing the bulk of the data.

In [ ]:
# Cell 19 — Outlier Removal (IQR Method)
q_low  = raw_data['Estimated_Deliveries'].quantile(0.25)
q_high = raw_data['Estimated_Deliveries'].quantile(0.75)
iqr_range = q_high - q_low

fence_low  = q_low  - 1.5 * iqr_range
fence_high = q_high + 1.5 * iqr_range

rows_before = len(raw_data)
cleaned_df  = raw_data[
    (raw_data['Estimated_Deliveries'] >= fence_low) &
    (raw_data['Estimated_Deliveries'] <= fence_high)
].copy()
rows_after = len(cleaned_df)

print(f'IQR range        : [{fence_low:.1f}, {fence_high:.1f}]')
print(f'Rows before      : {rows_before}')
print(f'Rows removed     : {rows_before - rows_after}')
print(f'Rows remaining   : {rows_after}')

## Cell 20 — Feature Engineering

Raw columns do not always capture the full story. We derive two additional features that encode domain-level insight:

| Feature | Formula | Interpretation |
|---------|---------|----------------|
| `Supply_Gap` | `Production_Units − Estimated_Deliveries` | Measures inventory backlog or supply shortfall |
| `Delivery_Density` | `Estimated_Deliveries / Charging_Stations` | Captures how well infrastructure supports delivery volume |

These features add signal that the model cannot easily derive from raw columns alone.

In [ ]:
# Cell 20 — Feature Engineering
cleaned_df['Supply_Gap']       = cleaned_df['Production_Units'] - cleaned_df['Estimated_Deliveries']
cleaned_df['Delivery_Density'] = cleaned_df['Estimated_Deliveries'] / cleaned_df['Charging_Stations']

print('New features added:')
print(cleaned_df[['Supply_Gap', 'Delivery_Density']].describe().round(2))

## Cell 21 — Standard ML Pipeline: Preprocessing & Train/Test Split

We define the feature matrix `X` and target `y`, then split into 80% training and 20% test sets using a **random split** (records treated independently).

Preprocessing is handled via a `ColumnTransformer` inside a `Pipeline`:
- Numerical features → `StandardScaler` (zero mean, unit variance)
- Categorical features → `OneHotEncoder` (binary indicator columns)

In [ ]:
# Cell 21 — Standard ML Pipeline: Preprocessing & Train/Test Split
delivery_target = cleaned_df['Estimated_Deliveries']
feature_matrix  = cleaned_df.drop(
    columns=['Estimated_Deliveries', 'Timestamp', 'Production_Units', 'CO2_Saved_tons']
)

num_cols = [
    'Year', 'Month', 'Avg_Price_USD', 'Battery_Capacity_kWh',
    'Range_km', 'Charging_Stations', 'Supply_Gap', 'Delivery_Density'
]
cat_cols = ['Region', 'Model', 'Source_Type']

col_transformer = ColumnTransformer(transformers=[
    ('scale',  StandardScaler(),                   num_cols),
    ('encode', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

feat_train, feat_test, lbl_train, lbl_test = train_test_split(
    feature_matrix, delivery_target, test_size=0.2, random_state=7
)

print(f'Training samples : {len(feat_train)}')
print(f'Test samples     : {len(feat_test)}')

## Cell 22 — Standard ML Pipeline: GridSearchCV & Model Selection

We test four regression algorithms under a unified `GridSearchCV` with **5-fold cross-validation** and `r2` as the scoring metric:

| Model | Regularization | Hyperparameter |
|-------|---------------|----------------|
| Linear Regression | None | — |
| Lasso | L1 | `alpha` ∈ {0.001, 0.01, 0.1, 1, 10} |
| Ridge | L2 | `alpha` ∈ {0.001, 0.01, 0.1, 1, 10} |
| Random Forest | Ensemble | `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf` |

The grid search automatically selects the configuration with the highest average cross-validated R².

In [ ]:
# Cell 22 — Standard ML Pipeline: GridSearchCV & Model Selection
base_pipeline = Pipeline(steps=[
    ('transform', col_transformer),
    ('model',     LinearRegression())
])

search_space = [
    {'model': [LinearRegression()]},
    {
        'model': [Lasso(max_iter=2000)],
        'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
    },
    {
        'model': [Ridge()],
        'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
    },
    {
        'model': [RandomForestRegressor(random_state=7)],
        'model__n_estimators':     [100, 200],
        'model__max_depth':        [3, 4, 5],
        'model__min_samples_split':[15, 20],
        'model__min_samples_leaf': [10, 15]
    }
]

cv_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=search_space,
    cv=5, scoring='r2', n_jobs=-1
)
cv_search.fit(feat_train, lbl_train)

champion_model = cv_search.best_estimator_
print(f'Best model       : {cv_search.best_estimator_.named_steps["model"].__class__.__name__}')
print(f'Best params      : {cv_search.best_params_}')
print(f'Best CV R²       : {cv_search.best_score_:.4f}')

## Cell 23 — Standard ML Pipeline: Evaluation & Visualization

We evaluate the best model on both training and held-out test data. Three metrics are reported:
- **R²** — proportion of variance explained (higher = better, max 1.0)
- **RMSE** — root mean squared error in the original delivery units
- **MSE** — mean squared error (square of RMSE)

Line plots compare actual vs predicted values to visually assess fit quality.

In [ ]:
# Cell 23 — Standard ML Pipeline: Evaluation & Visualization

# — Training metrics
train_preds = champion_model.predict(feat_train)
print('── Training Performance ──')
print(f'R²   : {r2_score(lbl_train, train_preds):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(lbl_train, train_preds)):.2f}')
print(f'MSE  : {mean_squared_error(lbl_train, train_preds):.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lbl_train.values, label='Actual',    alpha=0.8, linewidth=1.5, color='royalblue')
ax.plot(train_preds,       label='Predicted', linestyle='--', alpha=0.8, linewidth=1.5, color='darkorange')
ax.set_title('Standard ML — Training Set: Actual vs Predicted', fontsize=13)
ax.set_xlabel('Sample Index')
ax.set_ylabel('Estimated Deliveries')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

# — Test metrics
test_preds = champion_model.predict(feat_test)
print('\n── Test Performance ──')
print(f'R²   : {r2_score(lbl_test, test_preds):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(lbl_test, test_preds)):.2f}')
print(f'MSE  : {mean_squared_error(lbl_test, test_preds):.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lbl_test.values, label='Actual',    alpha=0.8, linewidth=1.5, color='royalblue')
ax.plot(test_preds,       label='Predicted', linestyle='--', alpha=0.8, linewidth=1.5, color='darkorange')
ax.set_title('Standard ML — Test Set: Actual vs Predicted', fontsize=13)
ax.set_xlabel('Sample Index')
ax.set_ylabel('Estimated Deliveries')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Cell 24 — Time-Series Pipeline: Data Preparation & Lag Features

For the time-series track, we must respect the **temporal ordering** of records. A random split would allow training on data from future months — an unrealistic advantage.

We sort by `Timestamp` and engineer three new temporal features:

| Feature | Description |
|---------|-------------|
| `lag_1` | Deliveries 1 period ago |
| `lag_2` | Deliveries 2 periods ago |
| `lag_3` | Deliveries 3 periods ago |
| `RollingAvg_3M` | 3-period rolling mean (shifted to avoid leakage) |

The **chronological 80/20 split** means the first 80% of time-ordered records are for training and the last 20% simulate genuine future forecasting.

In [ ]:
# Cell 24 — Time-Series Pipeline: Data Preparation & Lag Features
ts_df = cleaned_df.copy()
ts_df['Timestamp'] = pd.to_datetime(ts_df['Timestamp'])
ts_df = ts_df.sort_values('Timestamp').set_index('Timestamp')

# Lag features (t-1, t-2, t-3)
for lag in [1, 2, 3]:
    col = f'lag_{lag}'
    ts_df[col] = ts_df['Estimated_Deliveries'].shift(lag)
    ts_df[col].fillna(ts_df[col].mean(), inplace=True)

# Rolling 3-period mean (shifted by 1 to prevent data leakage)
ts_df['RollingAvg_3M'] = (
    ts_df['Estimated_Deliveries'].shift(1).rolling(window=3).mean()
)
ts_df['RollingAvg_3M'].fillna(ts_df['RollingAvg_3M'].mean(), inplace=True)
ts_df.dropna(inplace=True)

# Chronological split
ts_features = ts_df.drop(columns=['Estimated_Deliveries', 'CO2_Saved_tons', 'Production_Units'])
ts_target   = ts_df['Estimated_Deliveries']

cut = int(len(ts_features) * 0.8)
X_ts_train, X_ts_test = ts_features.iloc[:cut], ts_features.iloc[cut:]
y_ts_train, y_ts_test = ts_target.iloc[:cut],   ts_target.iloc[cut:]
train_dates, test_dates = X_ts_train.index, X_ts_test.index

print(f'Training period  : {train_dates.min().date()} → {train_dates.max().date()}  ({len(X_ts_train)} records)')
print(f'Test period      : {test_dates.min().date()} → {test_dates.max().date()}  ({len(X_ts_test)} records)')

## Cell 25 — Time-Series Pipeline: GridSearchCV with TimeSeriesSplit

Instead of k-fold cross-validation (which shuffles folds randomly), we use **`TimeSeriesSplit(n_splits=3)`**. This ensures that in each fold, the validation window always comes *after* the training window — correctly mimicking real deployment conditions.

The same four model families from the standard pipeline are evaluated here, with identical search spaces.

In [ ]:
# Cell 25 — Time-Series Pipeline: GridSearchCV with TimeSeriesSplit
ts_num_cols = [
    'Year', 'Month', 'Avg_Price_USD', 'Charging_Stations',
    'Battery_Capacity_kWh', 'Range_km',
    'lag_1', 'lag_2', 'lag_3', 'RollingAvg_3M',
    'Supply_Gap', 'Delivery_Density'
]
ts_cat_cols = ['Region', 'Model', 'Source_Type']

ts_preprocessor = ColumnTransformer(transformers=[
    ('scale',  StandardScaler(),                      ts_num_cols),
    ('encode', OneHotEncoder(handle_unknown='ignore'), ts_cat_cols)
])

ts_pipeline = Pipeline(steps=[
    ('transform', ts_preprocessor),
    ('model',     LinearRegression())
])

ts_search_space = [
    {'model': [LinearRegression()]},
    {
        'model': [Lasso(max_iter=5000, random_state=7)],
        'model__alpha': [0.1, 1.0, 10.0, 50.0, 100.0, 500.0]
    },
    {
        'model': [Ridge(random_state=7)],
        'model__alpha': [0.1, 1.0, 10.0, 50.0, 100.0, 500.0]
    },
    {
        'model': [RandomForestRegressor(random_state=7)],
        'model__n_estimators':     [100, 200],
        'model__max_depth':        [3, 4, 5],
        'model__min_samples_split':[15, 20],
        'model__min_samples_leaf': [10, 15]
    }
]

ts_cv = TimeSeriesSplit(n_splits=3)
ts_grid_search = GridSearchCV(
    estimator=ts_pipeline,
    param_grid=ts_search_space,
    cv=ts_cv, scoring='r2', n_jobs=-1
)
ts_grid_search.fit(X_ts_train, y_ts_train)

best_ts_model = ts_grid_search.best_estimator_
print(f'Best TS model    : {ts_grid_search.best_estimator_.named_steps["model"].__class__.__name__}')
print(f'Best TS params   : {ts_grid_search.best_params_}')
print(f'Best TS CV R²    : {ts_grid_search.best_score_:.4f}')

## Cell 26 — Time-Series Pipeline: Evaluation & Visualization

We evaluate the best time-series model on both periods. The **test plot** is the most important — it shows how well the model forecasts delivery volumes for months it has never seen during training or hyperparameter selection.

In [ ]:
# Cell 26 — Time-Series Pipeline: Evaluation & Visualization

# — Training metrics
ts_train_preds = best_ts_model.predict(X_ts_train)
print('── Time-Series Training Performance ──')
print(f'R²   : {r2_score(y_ts_train, ts_train_preds):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_ts_train, ts_train_preds)):.2f}')
print(f'MSE  : {mean_squared_error(y_ts_train, ts_train_preds):.2f}')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_dates, y_ts_train.values, label='Actual Deliveries', color='royalblue',  linewidth=2)
ax.plot(train_dates, ts_train_preds,     label='Model Forecast',    color='darkorange', linestyle='--', linewidth=2)
ax.set_title('Time-Series Pipeline — Training Period Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Estimated Deliveries')
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# — Test metrics
ts_test_preds = best_ts_model.predict(X_ts_test)
print('\n── Time-Series Test Performance ──')
print(f'R²   : {r2_score(y_ts_test, ts_test_preds):.4f}')
print(f'RMSE : {np.sqrt(mean_squared_error(y_ts_test, ts_test_preds)):.2f}')
print(f'MSE  : {mean_squared_error(y_ts_test, ts_test_preds):.2f}')

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test_dates, y_ts_test.values, label='Actual Deliveries', color='royalblue',  linewidth=2)
ax.plot(test_dates, ts_test_preds,     label='Model Forecast',    color='darkorange', linestyle='--', linewidth=2)
ax.set_title('Time-Series Pipeline — Out-of-Sample Forecast', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Estimated Deliveries')
ax.legend(fontsize=11)
ax.grid(True, linestyle=':', alpha=0.5)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Cell 27 — Project Conclusion & Key Findings

---

### Performance Summary

| Pipeline             | Split Strategy            | Train R² | Test R² | Train RMSE | Test RMSE |
| -------------------- | ------------------------- | -------- | ------- | ---------- | --------- |
| Standard ML Pipeline | Random 80/20 Split        | 0.8888   | 0.8481  | 1280.74    | 1528.71   |
| Time-Series Pipeline | Chronological 80/20 Split | 0.8786   | 0.8541  | 1361.02    | 1399.54   |

Both modelling approaches achieved strong predictive performance, with test R² scores above 0.84. While the Standard ML Pipeline produced a slightly higher training score, the Time-Series Pipeline achieved better performance on unseen test data, indicating stronger forecasting capability.

---

### Best Model Selection

Multiple regression models were evaluated using GridSearchCV, including:

* Linear Regression
* Ridge Regression
* Lasso Regression
* Random Forest Regressor

For the time-series forecasting task, the best-performing model was:

* **Model:** Random Forest Regressor
* **Maximum Depth:** 5
* **Minimum Samples Split:** 15
* **Minimum Samples Leaf:** 10
* **Number of Trees:** 200

The model achieved a cross-validation R² score of **0.8723**, demonstrating stable performance across multiple chronological validation folds.

---

### Model Performance Analysis

The training and testing metrics remain reasonably close across both pipelines.

For the Time-Series Pipeline:

* Training R² = **0.8786**
* Testing R² = **0.8541**

For the Standard ML Pipeline:

* Training R² = **0.8888**
* Testing R² = **0.8481**

The relatively small difference between training and testing performance suggests that the models have learned meaningful patterns from the dataset without severe overfitting.

---

### Importance of Time-Series Forecasting

Forecasting future deliveries requires preserving the natural order of observations. To address this requirement, a dedicated time-series workflow was implemented.

The forecasting pipeline included:

* Chronological data sorting
* Lag feature generation (t−1, t−2, t−3)
* Rolling average feature creation
* TimeSeriesSplit cross-validation
* Future-period evaluation using an out-of-sample test set

This approach provides a more realistic estimate of how the model would perform when predicting future delivery volumes.

---

### Feature Engineering Contributions

Several engineered features were introduced to improve predictive performance:

#### Supply_Gap

Difference between production volume and actual deliveries.

```text
Supply_Gap = Production_Units - Estimated_Deliveries
```

#### Delivery_Density

Measures delivery volume relative to available charging infrastructure.

```text
Delivery_Density = Estimated_Deliveries / Charging_Stations
```

#### Lag Features

Historical delivery observations used to capture temporal dependencies.

```text
lag_1, lag_2, lag_3
```

#### Rolling Average

A moving average feature used to smooth short-term fluctuations and identify trends.

```text
RollingAvg_3M
```

These engineered variables helped the model capture operational and temporal relationships that were not directly available in the original dataset.

---

### Assignment Objectives Covered

| Objective                       | Status    |
| ------------------------------- | --------- |
| Data Loading & Inspection       | Completed |
| Data Preprocessing              | Completed |
| Exploratory Data Analysis (EDA) | Completed |
| Feature Engineering             | Completed |
| Regression Modelling            | Completed |
| Hyperparameter Tuning           | Completed |
| Pipeline Development            | Completed |
| Time-Series Forecasting         | Completed |
| Model Evaluation                | Completed |

---

### Final Conclusion

This project successfully implemented an end-to-end machine learning pipeline for forecasting Tesla vehicle deliveries using historical production, pricing, infrastructure and operational data from 2015 to 2025.

The workflow covered data preprocessing, exploratory data analysis, feature engineering, regression modelling, hyperparameter optimization and time-series forecasting. Multiple regression algorithms were evaluated, and Random Forest Regressor emerged as the most effective model for capturing complex relationships within the dataset.

The final results demonstrate that machine learning techniques, combined with carefully engineered temporal features, can provide accurate delivery forecasts and valuable insights for future planning. The strong performance of the forecasting pipeline confirms the usefulness of both feature engineering and time-aware validation strategies when working with real-world delivery and sales datasets.